# Module 2: EIA Knowledge Base with RAG

## Learning Objectives

By completing this notebook, you will:
1. Build a vector database of historical EIA petroleum reports
2. Implement semantic search for relevant report sections
3. Create a RAG (Retrieval-Augmented Generation) system for commodity research
4. Query historical patterns and trends using natural language
5. Compare LLM answers with and without retrieval context

## Prerequisites

- Completed Module 1 (Report Processing)
- Understanding of vector embeddings
- OpenAI or Anthropic API access
- Basic knowledge of ChromaDB or similar vector databases

## Estimated Time: 90-105 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Completed Module 1 (Report Processing)",
    "Understanding of vector embeddings",
    "OpenAI or Anthropic API access",
    "Basic knowledge of ChromaDB or similar vector databases"
])

## 1. Understanding RAG for Commodity Research

**Problem:** LLMs don't know about recent commodity reports or your private research.

**Solution:** RAG (Retrieval-Augmented Generation)
1. Store reports in vector database (embeddings)
2. User asks a question
3. Retrieve relevant report sections
4. Feed to LLM as context
5. LLM generates answer based on actual data

**Use Cases:**
- "What were crude oil inventories during hurricanes last year?"
- "How did refinery utilization compare to 5-year averages?"
- "Find all mentions of production cuts"

In [ ]:
# Standard library imports
import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import matplotlib.pyplot as plt

# Vector database
import chromadb
from chromadb.config import Settings

# LLM imports
import openai
from anthropic import Anthropic

# Text processing
from sentence_transformers import SentenceTransformer

# Configuration
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# Initialize clients
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

# Data directories
DATA_DIR = Path('data/eia_reports')
CHROMA_DIR = Path('data/chroma_db')
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")
print(f"   Data directory: {DATA_DIR.absolute()}")
print(f"   Vector DB directory: {CHROMA_DIR.absolute()}")
print(f"   LLM clients: {'OpenAI' if openai_client else ''} {'Claude' if anthropic_client else ''}")

## 2. Setting Up Vector Database

We'll use ChromaDB - a lightweight vector database perfect for RAG applications.

**Key Concepts:**
- **Embeddings**: Vector representations of text (capture semantic meaning)
- **Collections**: Organized groups of embeddings
- **Similarity Search**: Find vectors close to query vector
- **Metadata**: Store document info alongside vectors

In [ ]:
# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Create or get collection for EIA reports
collection_name = "eia_petroleum_reports"

try:
    # Try to get existing collection
    collection = chroma_client.get_collection(name=collection_name)
    print(f"✅ Loaded existing collection: {collection_name}")
    print(f"   Documents in collection: {collection.count()}")
except:
    # Create new collection
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"description": "EIA Weekly Petroleum Status Reports"}
    )
    print(f"✅ Created new collection: {collection_name}")

# Initialize embedding model
# Using sentence-transformers (runs locally, no API cost)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ Loaded embedding model: all-MiniLM-L6-v2")
print(f"   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

## 3. Chunking and Indexing Reports

To make reports searchable, we need to:
1. Split into chunks (embeddings work best on ~500 token chunks)
2. Generate embeddings for each chunk
3. Store in vector database with metadata

**Chunking Strategy for EIA Reports:**
- Split by paragraph
- Keep section headers with chunks (context)
- Overlap chunks slightly to avoid losing context at boundaries

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping chunks.
    
    Parameters:
    -----------
    text : str
        Text to chunk
    chunk_size : int
        Approximate characters per chunk
    overlap : int
        Overlap between chunks
        
    Returns:
    --------
    List[str]
        List of text chunks
    """
    # Split by paragraphs first
    paragraphs = text.split('\n\n')
    
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        # If adding paragraph exceeds chunk size, save current and start new
        if len(current_chunk) + len(para) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # Start new chunk with overlap from previous
            current_chunk = current_chunk[-overlap:] + "\n\n" + para
        else:
            current_chunk += "\n\n" + para if current_chunk else para
    
    # Add final chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks

def index_report(report_text: str, report_date: str, collection) -> int:
    """
    Chunk report, generate embeddings, and add to vector database.
    
    Parameters:
    -----------
    report_text : str
        Full report text
    report_date : str
        Date of report (YYYY-MM-DD)
    collection
        ChromaDB collection
        
    Returns:
    --------
    int
        Number of chunks indexed
    """
    # Chunk the text
    chunks = chunk_text(report_text)
    
    if not chunks:
        return 0
    
    # Generate embeddings
    embeddings = embedding_model.encode(chunks, show_progress_bar=False)
    
    # Prepare for insertion
    ids = [f"{report_date}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [
        {
            'report_date': report_date,
            'chunk_id': i,
            'total_chunks': len(chunks)
        }
        for i in range(len(chunks))
    ]
    
    # Add to collection
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        documents=chunks,
        metadatas=metadatas
    )
    
    return len(chunks)

# Example: Chunk a sample text
sample_text = """
U.S. commercial crude oil inventories increased by 1.2 million barrels from the previous week. 
At 439.5 million barrels, U.S. crude oil inventories are about 2% above the five-year average.

Total motor gasoline inventories decreased by 3.1 million barrels last week and are approximately 
3% below the five-year average. Finished gasoline inventories decreased while blending components 
inventories increased.

Distillate fuel inventories decreased by 1.8 million barrels last week and are about 8% below 
the five-year average.
"""

sample_chunks = chunk_text(sample_text, chunk_size=200, overlap=30)
print(f"Sample text chunking:")
print(f"  Original length: {len(sample_text)} characters")
print(f"  Number of chunks: {len(sample_chunks)}")
print(f"\nChunks:")
for i, chunk in enumerate(sample_chunks):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)

### 💡 Exercise 3.1: Index Historical Reports

**Task:** Load all EIA reports from the data directory and index them in the vector database.

**Requirements:**
1. Find all PDF files in DATA_DIR
2. Extract text from each (use Module 1 extraction code)
3. Index each report using `index_report()`
4. Track success/failure for each
5. Print summary statistics

**Hints:**
- Use pathlib to find PDFs: `DATA_DIR.glob('*.pdf')`
- Reuse extraction functions from Module 1
- Handle errors gracefully

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
# Import extraction function from Module 1
import pdfplumber

def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract full text from PDF."""
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n\n"
    except Exception as e:
        print(f"Error extracting {pdf_path.name}: {e}")
    return text

# Find all EIA PDFs
if DATA_DIR.exists():
    pdf_files = list(DATA_DIR.glob('eia_petroleum_*.pdf'))
    print(f"Found {len(pdf_files)} EIA report PDFs\n")
    
    if pdf_files:
        indexed_count = 0
        total_chunks = 0
        
        for pdf_path in pdf_files:
            # Extract date from filename
            # Format: eia_petroleum_YYYY_MM_DD.pdf
            filename = pdf_path.stem
            parts = filename.split('_')
            if len(parts) >= 5:
                report_date = f"{parts[2]}-{parts[3]}-{parts[4]}"
            else:
                report_date = 'unknown'
            
            print(f"Indexing {pdf_path.name} ({report_date})...", end=' ')
            
            # Extract and index
            text = extract_text_from_pdf(pdf_path)
            if text:
                chunks = index_report(text, report_date, collection)
                print(f"✅ {chunks} chunks")
                indexed_count += 1
                total_chunks += chunks
            else:
                print("❌ No text extracted")
        
        print("\n" + "="*60)
        print(f"Indexing Complete")
        print("="*60)
        print(f"  Reports indexed: {indexed_count}/{len(pdf_files)}")
        print(f"  Total chunks: {total_chunks}")
        print(f"  Collection size: {collection.count()}")
    else:
        print("⚠️  No PDF files found. Download reports first using Module 1 code.")
else:
    print(f"⚠️  Data directory not found: {DATA_DIR}")

## 4. Semantic Search

Now that reports are indexed, we can search them using natural language queries.

**How it works:**
1. Convert query to embedding vector
2. Find most similar chunks in database (cosine similarity)
3. Return top K results with metadata

**Key Advantage:** Finds semantically similar content, not just keyword matches.

In [ ]:
def semantic_search(query: str, collection, top_k: int = 5) -> List[Dict]:
    """
    Search vector database for relevant report sections.
    
    Parameters:
    -----------
    query : str
        Natural language query
    collection
        ChromaDB collection
    top_k : int
        Number of results to return
        
    Returns:
    --------
    List[Dict]
        List of matching chunks with metadata and scores
    """
    # Generate query embedding
    query_embedding = embedding_model.encode([query])[0]
    
    # Search collection
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    
    # Format results
    formatted_results = []
    if results['ids'] and len(results['ids']) > 0:
        for i in range(len(results['ids'][0])):
            formatted_results.append({
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i] if 'distances' in results else None
            })
    
    return formatted_results

# Test semantic search
if collection.count() > 0:
    test_query = "What were crude oil inventory levels?"
    print(f"Query: '{test_query}'\n")
    
    results = semantic_search(test_query, collection, top_k=3)
    
    print(f"Found {len(results)} relevant chunks:\n")
    for i, result in enumerate(results):
        print(f"--- Result {i+1} ---")
        print(f"Date: {result['metadata'].get('report_date', 'unknown')}")
        if result['distance']:
            print(f"Similarity: {1 - result['distance']:.3f}")
        print(f"Text: {result['text'][:200]}...")
        print()
else:
    print("⚠️  Collection is empty. Index some reports first.")

### 💡 Exercise 4.1: Explore Different Queries

**Task:** Run semantic search on various commodity-related queries and analyze result quality.

**Queries to test:**
1. "refinery utilization rates"
2. "gasoline demand trends"
3. "hurricane impact on production"
4. "inventory builds versus draws"
5. "seasonal patterns"

**For each query:**
- Run semantic search
- Check if top result is actually relevant
- Note the similarity scores
- Identify any false positives

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
if collection.count() > 0:
    test_queries = [
        "refinery utilization rates",
        "gasoline demand trends",
        "hurricane impact on production",
        "inventory builds versus draws",
        "seasonal patterns"
    ]
    
    print("Testing semantic search quality:\n")
    print("="*70)
    
    for query in test_queries:
        print(f"\nQuery: '{query}'")
        print("-"*70)
        
        results = semantic_search(query, collection, top_k=2)
        
        if results:
            top_result = results[0]
            similarity = 1 - top_result['distance'] if top_result['distance'] else 0
            
            print(f"Top Result (similarity: {similarity:.3f}):")
            print(f"  Date: {top_result['metadata'].get('report_date', 'unknown')}")
            print(f"  Text: {top_result['text'][:150]}...")
            
            # Quality assessment
            if similarity > 0.7:
                print("  ✅ High confidence match")
            elif similarity > 0.5:
                print("  ⚠️  Moderate confidence - verify relevance")
            else:
                print("  ❌ Low confidence - may not be relevant")
        else:
            print("  No results found")
    
    print("\n" + "="*70)
else:
    print("⚠️  Collection is empty")

## 5. Building the RAG System

Now we combine retrieval with generation. The LLM answers questions using retrieved context.

**RAG Pipeline:**
1. User asks question
2. Retrieve relevant chunks
3. Format as context for LLM
4. LLM generates answer citing sources
5. Return answer with references

In [ ]:
def rag_query(question: str, collection, top_k: int = 5) -> Dict[str, Any]:
    """
    Answer question using RAG.
    
    Parameters:
    -----------
    question : str
        User's question
    collection
        Vector database collection
    top_k : int
        Number of chunks to retrieve
        
    Returns:
    --------
    dict
        Answer, sources, and metadata
    """
    # Step 1: Retrieve relevant context
    search_results = semantic_search(question, collection, top_k=top_k)
    
    if not search_results:
        return {
            'answer': "No relevant information found in the knowledge base.",
            'sources': [],
            'context_used': False
        }
    
    # Step 2: Format context for LLM
    context_sections = []
    for i, result in enumerate(search_results):
        date = result['metadata'].get('report_date', 'unknown')
        context_sections.append(
            f"[Source {i+1} - Report Date: {date}]\n{result['text']}\n"
        )
    
    context = "\n".join(context_sections)
    
    # Step 3: Create prompt
    prompt = f"""
You are a commodity market analyst answering questions based on EIA petroleum reports.

Use the following report excerpts to answer the question. Cite sources by number.
If the information is not in the provided context, say so.

Context:
{context}

Question: {question}

Answer:
"""
    
    # Step 4: Generate answer
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-4",
                messages=[
                    {"role": "system", "content": "You are a precise commodity analyst. Cite sources."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0
            )
            answer = response.choices[0].message.content
        
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                temperature=0.0,
                messages=[{"role": "user", "content": prompt}]
            )
            answer = message.content[0].text
        
        else:
            answer = "No LLM configured"
        
        # Step 5: Package results
        return {
            'answer': answer,
            'sources': [
                {
                    'date': r['metadata'].get('report_date', 'unknown'),
                    'text': r['text'][:200] + "..."
                }
                for r in search_results
            ],
            'context_used': True,
            'num_sources': len(search_results)
        }
    
    except Exception as e:
        return {
            'answer': f"Error generating answer: {str(e)}",
            'sources': [],
            'context_used': False
        }

# Test RAG system
if collection.count() > 0:
    test_question = "What were the recent trends in crude oil inventories?"
    print(f"Question: {test_question}\n")
    print("="*70)
    
    result = rag_query(test_question, collection, top_k=3)
    
    print(f"Answer:\n{result['answer']}\n")
    print("="*70)
    print(f"\nSources Used: {result['num_sources']}")
    for i, source in enumerate(result['sources']):
        print(f"\n[{i+1}] {source['date']}")
        print(f"    {source['text']}")
else:
    print("⚠️  Collection is empty")

### 💡 Exercise 5.1: Compare RAG vs. Non-RAG Answers

**Task:** Ask the same question with and without retrieval context to see the difference.

**Requirements:**
1. Choose a specific question about recent commodity data
2. Get answer using RAG (with retrieval)
3. Get answer directly from LLM (without retrieval)
4. Compare accuracy and specificity
5. Document which answer is more useful and why

**Example question:** "What was crude oil inventory level in the most recent report?"

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
question = "What was the crude oil inventory level in the most recent EIA report?"

print("Comparing RAG vs Non-RAG Answers")
print("="*70)
print(f"Question: {question}\n")

# Answer WITH retrieval (RAG)
if collection.count() > 0:
    print("\n[1] WITH RETRIEVAL (RAG)")
    print("-"*70)
    rag_result = rag_query(question, collection, top_k=3)
    print(rag_result['answer'])
    print(f"\nSources: {rag_result['num_sources']} report excerpts")

# Answer WITHOUT retrieval (direct LLM)
print("\n\n[2] WITHOUT RETRIEVAL (Direct LLM)")
print("-"*70)

try:
    if openai_client:
        response = openai_client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are a commodity analyst."},
                {"role": "user", "content": question}
            ],
            temperature=0.0
        )
        direct_answer = response.choices[0].message.content
    elif anthropic_client:
        message = anthropic_client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=500,
            temperature=0.0,
            messages=[{"role": "user", "content": question}]
        )
        direct_answer = message.content[0].text
    else:
        direct_answer = "No LLM configured"
    
    print(direct_answer)
    print("\nSources: None (general knowledge only)")
    
except Exception as e:
    print(f"Error: {e}")

# Analysis
print("\n\n[ANALYSIS]")
print("="*70)
print("Key Differences:")
print("  • RAG answer uses specific data from actual reports")
print("  • Direct LLM likely gives generic/outdated information")
print("  • RAG provides dates and can cite sources")
print("  • Direct LLM may refuse to answer due to knowledge cutoff")
print("\nConclusion: RAG is essential for time-sensitive commodity data")

## 6. Advanced RAG Features

Production RAG systems often include:
- **Filtering**: Search within date ranges or specific categories
- **Reranking**: Use cross-encoder to improve result relevance
- **Hybrid search**: Combine semantic + keyword search
- **Citation tracking**: Link answers to specific report sections

In [ ]:
def rag_query_with_filters(question: str, 
                           collection, 
                           date_filter: Optional[Dict] = None,
                           top_k: int = 5) -> Dict[str, Any]:
    """
    RAG query with date filtering.
    
    Parameters:
    -----------
    question : str
        User question
    collection
        Vector database
    date_filter : Dict, optional
        Date range filter, e.g., {'start': '2024-01-01', 'end': '2024-03-31'}
    top_k : int
        Results to return
        
    Returns:
    --------
    dict
        Answer with filtered sources
    """
    # Generate query embedding
    query_embedding = embedding_model.encode([question])[0]
    
    # Build filter if provided
    where_filter = None
    if date_filter:
        where_filter = {}
        if 'start' in date_filter:
            where_filter['report_date'] = {'$gte': date_filter['start']}
        if 'end' in date_filter:
            if 'report_date' not in where_filter:
                where_filter['report_date'] = {}
            where_filter['report_date']['$lte'] = date_filter['end']
    
    # Search with filter
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        where=where_filter if where_filter else None
    )
    
    # Format and generate answer (same as rag_query)
    if not results['ids'] or len(results['ids'][0]) == 0:
        return {
            'answer': f"No information found{' in specified date range' if date_filter else ''}.",
            'sources': [],
            'context_used': False
        }
    
    # Build context
    context_sections = []
    sources = []
    for i in range(len(results['ids'][0])):
        text = results['documents'][0][i]
        metadata = results['metadatas'][0][i]
        date = metadata.get('report_date', 'unknown')
        
        context_sections.append(f"[Source {i+1} - {date}]\n{text}\n")
        sources.append({'date': date, 'text': text[:200] + "..."})
    
    context = "\n".join(context_sections)
    
    # Generate answer
    prompt = f"""
Answer this question using the provided EIA report excerpts. Cite sources by number.

Context:
{context}

Question: {question}

Answer:
"""
    
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            answer = response.choices[0].message.content
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                temperature=0.0,
                messages=[{"role": "user", "content": prompt}]
            )
            answer = message.content[0].text
        else:
            answer = "No LLM configured"
        
        return {
            'answer': answer,
            'sources': sources,
            'context_used': True,
            'num_sources': len(sources),
            'filter_applied': date_filter is not None
        }
    
    except Exception as e:
        return {'answer': f"Error: {e}", 'sources': [], 'context_used': False}

# Test filtered query
if collection.count() > 0:
    print("Testing filtered RAG query:\n")
    
    # Get recent reports only (last 2 months)
    two_months_ago = (datetime.now() - timedelta(days=60)).strftime('%Y-%m-%d')
    
    question = "What were crude oil inventory trends?"
    date_filter = {'start': two_months_ago}
    
    print(f"Question: {question}")
    print(f"Filter: Reports after {two_months_ago}\n")
    print("="*70)
    
    result = rag_query_with_filters(question, collection, date_filter=date_filter, top_k=3)
    
    print(f"Answer:\n{result['answer']}\n")
    print("="*70)
    print(f"\nSources: {result['num_sources']} (filtered)")
else:
    print("⚠️  Collection is empty")

### 💡 Exercise 6.1: Build a Research Assistant

**Task:** Create an interactive research assistant that answers multiple related questions.

**Requirements:**
1. Define a research topic (e.g., "refinery operations")
2. Create 3-5 related questions
3. Use RAG to answer each question
4. Synthesize findings into a brief research note
5. Include source citations

**Example topic:** "Impact of refinery maintenance on gasoline supply"
**Example questions:**
- What were refinery utilization rates?
- How did gasoline inventories change?
- Were there any mentions of maintenance?

**Output:** Professional research note with citations

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------
if collection.count() > 0:
    research_topic = "Gasoline Market Dynamics"
    questions = [
        "What were gasoline inventory levels and trends?",
        "How did refinery utilization rates affect gasoline supply?",
        "Were there any demand indicators mentioned?"
    ]
    
    print("="*70)
    print(f"RESEARCH NOTE: {research_topic}")
    print("="*70)
    print(f"Date: {datetime.now().strftime('%Y-%m-%d')}")
    print(f"Source: EIA Weekly Petroleum Status Reports\n")
    
    findings = []
    all_sources = []
    
    for i, question in enumerate(questions, 1):
        print(f"\n[Question {i}] {question}")
        print("-"*70)
        
        result = rag_query(question, collection, top_k=2)
        print(result['answer'])
        
        findings.append({
            'question': question,
            'answer': result['answer'],
            'sources': result['sources']
        })
        all_sources.extend(result['sources'])
    
    # Synthesis
    print("\n\n" + "="*70)
    print("SYNTHESIS")
    print("="*70)
    print("""
Based on recent EIA reports, gasoline market dynamics show:

1. Inventory levels reflect seasonal patterns and refinery activity
2. Refinery utilization rates directly impact product supply
3. Demand signals provide context for inventory changes

These factors should be monitored together for complete market picture.
    """)
    
    # Unique sources
    unique_dates = set([s['date'] for s in all_sources])
    print(f"\nSources Consulted: {len(unique_dates)} EIA reports")
    for date in sorted(unique_dates):
        print(f"  - EIA Weekly Petroleum Status Report, {date}")
    
    print("\n" + "="*70)
else:
    print("⚠️  Collection is empty")

## Summary

### Key Takeaways

1. **RAG Solves the Knowledge Gap** - LLMs can't know recent reports without retrieval

2. **Vector Databases Enable Semantic Search** - Find relevant content beyond keyword matching

3. **Chunking Strategy Matters** - Balance between context and specificity

4. **Citations Build Trust** - Always link answers to source documents

5. **Filtering Improves Relevance** - Date ranges and categories narrow results

### Skills Gained

✅ Building vector databases with ChromaDB  
✅ Generating embeddings for semantic search  
✅ Implementing RAG pipelines  
✅ Comparing RAG vs direct LLM answers  
✅ Adding filters and advanced features  
✅ Creating research assistants with LLMs  

### What's Next

In **Module 3, Notebook 1** (`01_news_sentiment.ipynb`), we'll analyze commodity news sentiment using:
- News API integration
- LLM-based sentiment analysis
- Correlation with price movements
- Building sentiment indicators

### Additional Resources

- **ChromaDB Documentation:** https://docs.trychroma.com/
- **RAG Best Practices:** https://www.anthropic.com/index/contextual-retrieval
- **Vector Database Comparison:** ChromaDB vs Pinecone vs Weaviate

---

**Next:** [Module 3 - Sentiment Analysis](../../module_03_sentiment/notebooks/01_news_sentiment.ipynb)